In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT = Path("/home/bannanakid/RCRL_on_RCMDP")
sys.path.insert(0, str(PROJECT))

from envs.half_cheetah_pitch_cost import HalfCheetahPitchCost
from configs.halfcheetah_safety_thresholds import PITCH_THRESHOLD, PITCH_SCALE, COST_VERSION

RUN = 1
FIRST_SEED = 1005
EPISODES = 50
EVALUATE_FREQ = 25
PERSISTENT_EPS = 0.1
UNCONSTRAINED_ACTOR_PATH = PROJECT / "models" / "HalfCheetahUnconstrained" / f"run{RUN}" / "Best_Unconstrained_actor"
PITCH_ACTOR_PATH = PROJECT / "models" / "HalfCheetahPitchCost" / f"run{RUN}" / "Best_PitchCost_actor"
OUTPUT = PROJECT / "plot_data" / "HalfCheetahPitchCost" / f"run{RUN}" / "evaluation"
OUTPUT.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT)
print("Pitch threshold:", PITCH_THRESHOLD)
print("Pitch scale:", PITCH_SCALE)
print("Cost version:", COST_VERSION)
print("Held-out seeds:", FIRST_SEED, "through", FIRST_SEED + EPISODES - 1)
print("Unconstrained actor:", UNCONSTRAINED_ACTOR_PATH)
print("Pitch actor:", PITCH_ACTOR_PATH)

In [ ]:
if PITCH_THRESHOLD is None or PITCH_SCALE is None:
    raise RuntimeError("PITCH_THRESHOLD and PITCH_SCALE must be finalized before evaluation")

if not UNCONSTRAINED_ACTOR_PATH.exists():
    raise FileNotFoundError(UNCONSTRAINED_ACTOR_PATH)

if not PITCH_ACTOR_PATH.exists():
    raise FileNotFoundError(
        f"{PITCH_ACTOR_PATH}\nNo feasible Best_PitchCost_actor was found. Check the pitch training evaluation outputs."
    )

print("Both actor files found")

In [ ]:
class ActorGaussian(nn.Module):
    def __init__(self, state_dim, hidden_width, action_dim, max_action):
        super().__init__()
        self.max_action = float(max_action)
        self.fc1 = nn.Linear(state_dim, hidden_width)
        self.fc2 = nn.Linear(hidden_width, hidden_width)
        self.fc3 = nn.Linear(hidden_width, hidden_width)
        self.mean_layer = nn.Linear(hidden_width, action_dim)
        self.log_std = nn.Parameter(torch.zeros(1, action_dim))
        self.activate_func = nn.ReLU()

    def forward(self, state):
        state = self.activate_func(self.fc1(state))
        state = self.activate_func(self.fc2(state))
        state = self.activate_func(self.fc3(state))
        return self.max_action * torch.tanh(self.mean_layer(state))


def unwrap_reset(reset_result):
    result = reset_result
    while isinstance(result, tuple):
        result = result[0]
    return np.asarray(result, dtype=np.float64)


def load_actor(path, env):
    state = torch.load(path, map_location="cpu")
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    if isinstance(state, dict) and "actor_state_dict" in state:
        state = state["actor_state_dict"]

    actor = ActorGaussian(
        int(state["fc1.weight"].shape[1]),
        int(state["fc1.weight"].shape[0]),
        int(state["mean_layer.weight"].shape[0]),
        float(env.action_space.high[0]),
    )
    actor.load_state_dict(state)
    actor.eval()
    return actor

In [ ]:
probe_env = HalfCheetahPitchCost()
unconstrained_actor = load_actor(UNCONSTRAINED_ACTOR_PATH, probe_env)
pitch_actor = load_actor(PITCH_ACTOR_PATH, probe_env)

print("Observation dimension:", probe_env.observation_space.shape[0])
print("Action dimension:", probe_env.action_space.shape[0])
print("Unconstrained parameters:", sum(p.numel() for p in unconstrained_actor.parameters()))
print("Pitch actor parameters:", sum(p.numel() for p in pitch_actor.parameters()))

In [ ]:
def collect_policy(actor, label, first_seed, episodes):
    env = HalfCheetahPitchCost()

    episode = []
    reset_seed = []
    step = []
    reward = []
    cost = []
    xpos = []
    torso_pitch = []
    torso_pitch_abs = []
    pitch_excess = []
    actions = []
    maximum_action_abs = []

    episode_rewards = []
    episode_costs = []
    episode_max_costs = []
    episode_lengths = []
    episode_net_displacements = []
    episode_absolute_displacements = []
    episode_traveled_distances = []

    for ep in range(episodes):
        ep_seed = first_seed + ep
        obs = unwrap_reset(env.reset(seed=ep_seed))
        start_x = float(env.sim.data.qpos[0])
        previous_x = start_x
        total_reward = 0.0
        total_cost = 0.0
        max_cost = 0.0
        traveled = 0.0
        t = 0
        done = False

        while not done:
            x = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)

            with torch.no_grad():
                action = actor(x).cpu().numpy().reshape(-1)

            action = np.clip(action, env.action_space.low, env.action_space.high)
            obs_next, r, c, truncated, terminated, info = env.step(action)
            done = bool(truncated or terminated)

            current_x = float(info.get("xpos", env.sim.data.qpos[0]))
            pitch = float(info.get("torso_pitch", env.sim.data.qpos[2]))
            pitch_abs = abs(pitch)
            excess = max(pitch_abs - float(PITCH_THRESHOLD), 0.0)

            traveled += abs(current_x - previous_x)
            previous_x = current_x

            episode.append(ep + 1)
            reset_seed.append(ep_seed)
            step.append(t)
            reward.append(float(r))
            cost.append(float(c))
            xpos.append(current_x)
            torso_pitch.append(pitch)
            torso_pitch_abs.append(pitch_abs)
            pitch_excess.append(excess)
            actions.append(np.asarray(action, dtype=np.float64).copy())
            maximum_action_abs.append(float(np.max(np.abs(action))))

            total_reward += float(r)
            total_cost += float(c)
            max_cost = max(max_cost, float(c))
            obs = obs_next
            t += 1

        end_x = float(env.sim.data.qpos[0])

        episode_rewards.append(total_reward)
        episode_costs.append(total_cost)
        episode_max_costs.append(max_cost)
        episode_lengths.append(t)
        episode_net_displacements.append(end_x - start_x)
        episode_absolute_displacements.append(abs(end_x - start_x))
        episode_traveled_distances.append(traveled)

        print(
            f"{label}: episode {ep + 1}/{episodes} seed={ep_seed} "
            f"reward={total_reward:.3f} total_cost={total_cost:.6f} max_cost={max_cost:.6f}"
        )

    return {
        "label": label,
        "episode": np.asarray(episode, dtype=np.int64),
        "reset_seed": np.asarray(reset_seed, dtype=np.int64),
        "step": np.asarray(step, dtype=np.int64),
        "reward": np.asarray(reward, dtype=np.float64),
        "cost": np.asarray(cost, dtype=np.float64),
        "xpos": np.asarray(xpos, dtype=np.float64),
        "torso_pitch": np.asarray(torso_pitch, dtype=np.float64),
        "torso_pitch_abs": np.asarray(torso_pitch_abs, dtype=np.float64),
        "pitch_excess": np.asarray(pitch_excess, dtype=np.float64),
        "actions": np.asarray(actions, dtype=np.float64),
        "maximum_action_abs": np.asarray(maximum_action_abs, dtype=np.float64),
        "episode_rewards": np.asarray(episode_rewards, dtype=np.float64),
        "episode_costs": np.asarray(episode_costs, dtype=np.float64),
        "episode_max_costs": np.asarray(episode_max_costs, dtype=np.float64),
        "episode_lengths": np.asarray(episode_lengths, dtype=np.int64),
        "episode_net_displacements": np.asarray(episode_net_displacements, dtype=np.float64),
        "episode_absolute_displacements": np.asarray(episode_absolute_displacements, dtype=np.float64),
        "episode_traveled_distances": np.asarray(episode_traveled_distances, dtype=np.float64),
    }

In [ ]:
unconstrained = collect_policy(
    unconstrained_actor,
    "Unconstrained",
    FIRST_SEED,
    EPISODES,
)

pitch_constrained = collect_policy(
    pitch_actor,
    "Pitch constrained",
    FIRST_SEED,
    EPISODES,
)

In [ ]:
def save_dataset(data, path):
    np.savez_compressed(
        path,
        episode=data["episode"],
        reset_seed=data["reset_seed"],
        step=data["step"],
        reward=data["reward"],
        cost=data["cost"],
        xpos=data["xpos"],
        torso_pitch=data["torso_pitch"],
        torso_pitch_abs=data["torso_pitch_abs"],
        pitch_excess=data["pitch_excess"],
        actions=data["actions"],
        maximum_action_abs=data["maximum_action_abs"],
        episode_rewards=data["episode_rewards"],
        episode_costs=data["episode_costs"],
        episode_max_costs=data["episode_max_costs"],
        episode_lengths=data["episode_lengths"],
        episode_net_displacements=data["episode_net_displacements"],
        episode_absolute_displacements=data["episode_absolute_displacements"],
        episode_traveled_distances=data["episode_traveled_distances"],
        pitch_threshold=np.asarray(PITCH_THRESHOLD, dtype=np.float64),
        pitch_scale=np.asarray(PITCH_SCALE, dtype=np.float64),
        cost_version=np.asarray(str(COST_VERSION)),
        first_seed=np.asarray(FIRST_SEED, dtype=np.int64),
        number_of_episodes=np.asarray(EPISODES, dtype=np.int64),
    )


save_dataset(unconstrained, OUTPUT / "unconstrained_on_pitch_env.npz")
save_dataset(pitch_constrained, OUTPUT / "pitch_constrained_evaluation.npz")

print("Saved:", OUTPUT / "unconstrained_on_pitch_env.npz")
print("Saved:", OUTPUT / "pitch_constrained_evaluation.npz")

In [ ]:
def longest_true_run(values):
    best = 0
    current = 0

    for value in values:
        if value:
            current += 1
            best = max(best, current)
        else:
            current = 0

    return best


def violation_events(values):
    values = np.asarray(values, dtype=bool)

    if len(values) == 0:
        return 0

    return int(values[0]) + int(np.sum(values[1:] & ~values[:-1]))


def episode_violation_arrays(data):
    max_runs = []
    events = []
    violated = []

    for ep in np.unique(data["episode"]):
        mask = data["episode"] == ep
        flags = data["torso_pitch_abs"][mask] > float(PITCH_THRESHOLD)
        max_runs.append(longest_true_run(flags))
        events.append(violation_events(flags))
        violated.append(bool(np.any(flags)))

    return (
        np.asarray(max_runs, dtype=np.int64),
        np.asarray(events, dtype=np.int64),
        np.asarray(violated, dtype=bool),
    )


def summarize_policy(data):
    flags = data["torso_pitch_abs"] > float(PITCH_THRESHOLD)
    max_runs, events, violated = episode_violation_arrays(data)

    return {
        "mean_episode_reward": float(np.mean(data["episode_rewards"])),
        "std_episode_reward": float(np.std(data["episode_rewards"])),
        "median_episode_reward": float(np.median(data["episode_rewards"])),
        "mean_episode_total_cost": float(np.mean(data["episode_costs"])),
        "median_episode_total_cost": float(np.median(data["episode_costs"])),
        "maximum_one_step_cost": float(np.max(data["cost"])),
        "mean_absolute_pitch": float(np.mean(data["torso_pitch_abs"])),
        "pitch_p95": float(np.percentile(data["torso_pitch_abs"], 95)),
        "pitch_p99": float(np.percentile(data["torso_pitch_abs"], 99)),
        "maximum_absolute_pitch": float(np.max(data["torso_pitch_abs"])),
        "step_violation_rate": float(np.mean(flags)),
        "episode_violation_frequency": float(np.mean(violated)),
        "longest_violation_run": int(np.max(max_runs)),
        "mean_longest_violation_run": float(np.mean(max_runs)),
        "mean_violation_events_per_episode": float(np.mean(events)),
        "mean_net_displacement": float(np.mean(data["episode_net_displacements"])),
        "mean_absolute_displacement": float(np.mean(data["episode_absolute_displacements"])),
        "mean_traveled_distance": float(np.mean(data["episode_traveled_distances"])),
        "mean_maximum_action_abs": float(np.mean(data["maximum_action_abs"])),
    }


unconstrained_summary = summarize_policy(unconstrained)
pitch_summary = summarize_policy(pitch_constrained)

summary = pd.DataFrame(
    {
        "Unconstrained": unconstrained_summary,
        "Pitch constrained": pitch_summary,
    }
)

summary

In [ ]:
comparison = summary.copy()
reference = comparison["Unconstrained"].astype(float)
candidate = comparison["Pitch constrained"].astype(float)

comparison["Absolute change"] = candidate - reference
comparison["Percent change"] = np.where(
    np.abs(reference) > 1e-12,
    100.0 * (candidate - reference) / np.abs(reference),
    np.nan,
)

comparison.to_csv(OUTPUT / "summary.csv")
display(comparison)

print("Saved:", OUTPUT / "summary.csv")

In [ ]:
reward_retention = 100.0 * pitch_summary["mean_episode_reward"] / unconstrained_summary["mean_episode_reward"]


def reduction(before, after):
    if abs(before) <= 1e-12:
        return np.nan
    return 100.0 * (before - after) / abs(before)


step_violation_reduction = reduction(
    unconstrained_summary["step_violation_rate"],
    pitch_summary["step_violation_rate"],
)

episode_violation_reduction = reduction(
    unconstrained_summary["episode_violation_frequency"],
    pitch_summary["episode_violation_frequency"],
)

total_cost_reduction = reduction(
    unconstrained_summary["mean_episode_total_cost"],
    pitch_summary["mean_episode_total_cost"],
)

p99_reduction = reduction(
    unconstrained_summary["pitch_p99"],
    pitch_summary["pitch_p99"],
)

print(f"Reward retention: {reward_retention:.2f}%")
print(f"Step violation reduction: {step_violation_reduction:.2f}%")
print(f"Episode violation-frequency reduction: {episode_violation_reduction:.2f}%")
print(f"Mean episode pitch-cost reduction: {total_cost_reduction:.2f}%")
print(f"Pitch P99 reduction: {p99_reduction:.2f}%")

In [ ]:
episode_table = pd.DataFrame(
    {
        "seed": np.arange(FIRST_SEED, FIRST_SEED + EPISODES),
        "unconstrained_reward": unconstrained["episode_rewards"],
        "pitch_constrained_reward": pitch_constrained["episode_rewards"],
        "unconstrained_total_cost": unconstrained["episode_costs"],
        "pitch_constrained_total_cost": pitch_constrained["episode_costs"],
        "unconstrained_max_cost": unconstrained["episode_max_costs"],
        "pitch_constrained_max_cost": pitch_constrained["episode_max_costs"],
        "unconstrained_net_displacement": unconstrained["episode_net_displacements"],
        "pitch_constrained_net_displacement": pitch_constrained["episode_net_displacements"],
    }
)

u_runs, u_events, u_violated = episode_violation_arrays(unconstrained)
p_runs, p_events, p_violated = episode_violation_arrays(pitch_constrained)

episode_table["unconstrained_longest_violation_run"] = u_runs
episode_table["pitch_constrained_longest_violation_run"] = p_runs
episode_table["unconstrained_violation_events"] = u_events
episode_table["pitch_constrained_violation_events"] = p_events
episode_table["unconstrained_any_violation"] = u_violated
episode_table["pitch_constrained_any_violation"] = p_violated

episode_table.to_csv(OUTPUT / "episode_comparison.csv", index=False)
display(episode_table)

print("Saved:", OUTPUT / "episode_comparison.csv")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(
    episode_table["seed"],
    episode_table["unconstrained_reward"],
    label="Unconstrained",
)
plt.plot(
    episode_table["seed"],
    episode_table["pitch_constrained_reward"],
    label="Pitch constrained",
)
plt.xlabel("Held-out reset seed")
plt.ylabel("Episode reward")
plt.title("Episode reward on identical held-out seeds")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT / "episode_reward_comparison.png", dpi=160)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.boxplot(
    [
        unconstrained["episode_rewards"],
        pitch_constrained["episode_rewards"],
    ],
    labels=["Unconstrained", "Pitch constrained"],
)
plt.ylabel("Episode reward")
plt.title("Held-out episode reward distribution")
plt.tight_layout()
plt.savefig(OUTPUT / "reward_boxplot.png", dpi=160)
plt.show()

In [ ]:
low = min(
    float(np.min(unconstrained["torso_pitch_abs"])),
    float(np.min(pitch_constrained["torso_pitch_abs"])),
)
high = max(
    float(np.max(unconstrained["torso_pitch_abs"])),
    float(np.max(pitch_constrained["torso_pitch_abs"])),
)
bins = np.linspace(low, high, 120)

plt.figure(figsize=(10, 5))
plt.hist(
    unconstrained["torso_pitch_abs"],
    bins=bins,
    density=True,
    alpha=0.5,
    label="Unconstrained",
)
plt.hist(
    pitch_constrained["torso_pitch_abs"],
    bins=bins,
    density=True,
    alpha=0.5,
    label="Pitch constrained",
)
plt.axvline(float(PITCH_THRESHOLD), linestyle="--", label="Pitch threshold")
plt.xlabel("Absolute torso pitch")
plt.ylabel("Density")
plt.title("Absolute torso-pitch distribution")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT / "pitch_distribution.png", dpi=160)
plt.show()

In [ ]:
def ecdf(values):
    x = np.sort(np.asarray(values, dtype=np.float64))
    y = np.arange(1, len(x) + 1, dtype=np.float64) / len(x)
    return x, y


u_x, u_y = ecdf(unconstrained["torso_pitch_abs"])
p_x, p_y = ecdf(pitch_constrained["torso_pitch_abs"])

plt.figure(figsize=(10, 5))
plt.plot(u_x, u_y, label="Unconstrained")
plt.plot(p_x, p_y, label="Pitch constrained")
plt.axvline(float(PITCH_THRESHOLD), linestyle="--", label="Pitch threshold")
plt.xlabel("Absolute torso pitch")
plt.ylabel("ECDF")
plt.title("Absolute torso-pitch ECDF")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT / "pitch_ecdf.png", dpi=160)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(
    episode_table["seed"],
    episode_table["unconstrained_total_cost"],
    label="Unconstrained",
)
plt.plot(
    episode_table["seed"],
    episode_table["pitch_constrained_total_cost"],
    label="Pitch constrained",
)
plt.xlabel("Held-out reset seed")
plt.ylabel("Episode total pitch cost")
plt.title("Pitch cost on identical held-out seeds")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT / "episode_total_cost_comparison.png", dpi=160)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(
    episode_table["seed"],
    episode_table["unconstrained_longest_violation_run"],
    label="Unconstrained",
)
plt.plot(
    episode_table["seed"],
    episode_table["pitch_constrained_longest_violation_run"],
    label="Pitch constrained",
)
plt.xlabel("Held-out reset seed")
plt.ylabel("Longest consecutive violation run")
plt.title("Longest pitch-threshold violation per episode")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT / "violation_run_comparison.png", dpi=160)
plt.show()

In [ ]:
def worst_episode_by_cost(data):
    index = int(np.argmax(data["episode_costs"]))
    return index + 1, int(data["reset_seed"][data["episode"] == index + 1][0])


u_worst_episode, u_worst_seed = worst_episode_by_cost(unconstrained)
p_worst_episode, p_worst_seed = worst_episode_by_cost(pitch_constrained)

print("Unconstrained worst cost episode:", u_worst_episode, "seed:", u_worst_seed)
print("Pitch-constrained worst cost episode:", p_worst_episode, "seed:", p_worst_seed)

In [ ]:
mask = unconstrained["episode"] == u_worst_episode

plt.figure(figsize=(11, 5))
plt.plot(
    unconstrained["step"][mask],
    unconstrained["torso_pitch_abs"][mask],
    label="Absolute pitch",
)
plt.axhline(float(PITCH_THRESHOLD), linestyle="--", label="Pitch threshold")
plt.xlabel("RL step")
plt.ylabel("Absolute torso pitch")
plt.title(f"Unconstrained worst pitch-cost episode, seed {u_worst_seed}")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT / "unconstrained_worst_pitch_episode.png", dpi=160)
plt.show()

In [ ]:
mask = pitch_constrained["episode"] == p_worst_episode

plt.figure(figsize=(11, 5))
plt.plot(
    pitch_constrained["step"][mask],
    pitch_constrained["torso_pitch_abs"][mask],
    label="Absolute pitch",
)
plt.axhline(float(PITCH_THRESHOLD), linestyle="--", label="Pitch threshold")
plt.xlabel("RL step")
plt.ylabel("Absolute torso pitch")
plt.title(f"Pitch-constrained worst pitch-cost episode, seed {p_worst_seed}")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT / "pitch_constrained_worst_pitch_episode.png", dpi=160)
plt.show()

In [ ]:
TRAINING_DATA = PROJECT / "data_train" / "HalfCheetahPitchCost" / f"run{RUN}"

reward_history_path = TRAINING_DATA / "evaluate_rewards.npy"
cost_history_path = TRAINING_DATA / "evaluate_costs.npy"
max_cost_history_path = TRAINING_DATA / "evaluate_max_costs.npy"

training_histories_exist = (
    reward_history_path.exists()
    and cost_history_path.exists()
    and max_cost_history_path.exists()
)

print("Training histories found:", training_histories_exist)
print("Training data directory:", TRAINING_DATA)

In [ ]:
if training_histories_exist:
    evaluation_rewards = np.load(reward_history_path)
    evaluation_costs = np.load(cost_history_path)
    evaluation_max_costs = np.load(max_cost_history_path)
    evaluation_episode = np.arange(len(evaluation_rewards)) * EVALUATE_FREQ

    print("Evaluation checkpoints:", len(evaluation_rewards))
    print("Last five rewards:", evaluation_rewards[-5:])
    print("Last five total costs:", evaluation_costs[-5:])
    print("Last five maximum costs:", evaluation_max_costs[-5:])

In [ ]:
if training_histories_exist:
    plt.figure(figsize=(10, 5))
    plt.plot(evaluation_episode, evaluation_rewards)
    plt.xlabel("Training episode")
    plt.ylabel("Evaluation reward")
    plt.title("Pitch-constrained training evaluation reward")
    plt.tight_layout()
    plt.savefig(OUTPUT / "training_evaluation_reward.png", dpi=160)
    plt.show()

In [ ]:
if training_histories_exist:
    plt.figure(figsize=(10, 5))
    plt.plot(evaluation_episode, evaluation_costs)
    plt.xlabel("Training episode")
    plt.ylabel("Evaluation total pitch cost")
    plt.title("Pitch-constrained training evaluation cost")
    plt.tight_layout()
    plt.savefig(OUTPUT / "training_evaluation_cost.png", dpi=160)
    plt.show()

In [ ]:
if training_histories_exist:
    plt.figure(figsize=(10, 5))
    plt.plot(evaluation_episode, evaluation_max_costs, label="Evaluation max cost")
    plt.axhline(PERSISTENT_EPS, linestyle="--", label=f"persistent_eps = {PERSISTENT_EPS}")
    plt.xlabel("Training episode")
    plt.ylabel("Maximum one-step pitch cost")
    plt.title("Pitch-constrained maximum evaluation cost")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT / "training_evaluation_max_cost.png", dpi=160)
    plt.show()

In [ ]:
result = pd.DataFrame(
    [
        {
            "metric": "Mean episode reward",
            "unconstrained": unconstrained_summary["mean_episode_reward"],
            "pitch_constrained": pitch_summary["mean_episode_reward"],
        },
        {
            "metric": "Mean episode total pitch cost",
            "unconstrained": unconstrained_summary["mean_episode_total_cost"],
            "pitch_constrained": pitch_summary["mean_episode_total_cost"],
        },
        {
            "metric": "Maximum one-step pitch cost",
            "unconstrained": unconstrained_summary["maximum_one_step_cost"],
            "pitch_constrained": pitch_summary["maximum_one_step_cost"],
        },
        {
            "metric": "Step violation rate",
            "unconstrained": unconstrained_summary["step_violation_rate"],
            "pitch_constrained": pitch_summary["step_violation_rate"],
        },
        {
            "metric": "Episode violation frequency",
            "unconstrained": unconstrained_summary["episode_violation_frequency"],
            "pitch_constrained": pitch_summary["episode_violation_frequency"],
        },
        {
            "metric": "Longest violation run",
            "unconstrained": unconstrained_summary["longest_violation_run"],
            "pitch_constrained": pitch_summary["longest_violation_run"],
        },
        {
            "metric": "Pitch P95",
            "unconstrained": unconstrained_summary["pitch_p95"],
            "pitch_constrained": pitch_summary["pitch_p95"],
        },
        {
            "metric": "Pitch P99",
            "unconstrained": unconstrained_summary["pitch_p99"],
            "pitch_constrained": pitch_summary["pitch_p99"],
        },
        {
            "metric": "Maximum absolute pitch",
            "unconstrained": unconstrained_summary["maximum_absolute_pitch"],
            "pitch_constrained": pitch_summary["maximum_absolute_pitch"],
        },
        {
            "metric": "Mean net displacement",
            "unconstrained": unconstrained_summary["mean_net_displacement"],
            "pitch_constrained": pitch_summary["mean_net_displacement"],
        },
    ]
)

result["absolute_change"] = result["pitch_constrained"] - result["unconstrained"]
result["percent_change"] = np.where(
    np.abs(result["unconstrained"]) > 1e-12,
    100.0 * result["absolute_change"] / np.abs(result["unconstrained"]),
    np.nan,
)

result.to_csv(OUTPUT / "final_heldout_result.csv", index=False)
display(result)

print()
print(f"Reward retention: {reward_retention:.2f}%")
print(f"Step violation reduction: {step_violation_reduction:.2f}%")
print(f"Episode violation-frequency reduction: {episode_violation_reduction:.2f}%")
print(f"Mean episode pitch-cost reduction: {total_cost_reduction:.2f}%")
print(f"Pitch P99 reduction: {p99_reduction:.2f}%")
print("Output directory:", OUTPUT)

In [ ]:
if "torso_pitch_raw" not in cal or "torso_pitch_raw" not in val:
    raise RuntimeError(
        "Recollect calibration and validation metrics with the wrapped-pitch collector before running this cell"
    )

PRE_EVENT_STEPS = 40
MIN_DETECTION_RATE = 0.90
MIN_MEDIAN_LEAD_STEPS = 5
CONTROL_DT = 0.05
CANDIDATE_THRESHOLDS = np.arange(0.25, 2.81, 0.05)


def inversion_events_and_normal_mask(data):
    episodes = np.asarray(data["episode"], dtype=np.int64)
    steps = np.asarray(data["step"], dtype=np.int64)
    raw = np.asarray(data["torso_pitch_raw"], dtype=np.float64)
    wrapped_abs = np.asarray(data["torso_pitch_abs"], dtype=np.float64)

    normal_mask = np.zeros(len(episodes), dtype=bool)
    events = []
    non_inversion_episodes = []

    for ep in np.unique(episodes):
        idx = np.flatnonzero(episodes == ep)

        ep_raw = raw[idx]
        ep_steps = steps[idx]

        baseline = float(ep_raw[0])
        excursion = np.abs(ep_raw - baseline)

        inversion_local = np.flatnonzero(excursion >= np.pi)

        if len(inversion_local) == 0:
            normal_mask[idx] = True
            non_inversion_episodes.append(int(ep))
            continue

        event_local = int(inversion_local[0])
        event_global = int(idx[event_local])

        hazard_start_local = max(
            0,
            event_local - PRE_EVENT_STEPS,
        )

        if hazard_start_local > 0:
            normal_mask[idx[:hazard_start_local]] = True

        events.append(
            {
                "episode": int(ep),
                "event_global": event_global,
                "event_local": event_local,
                "hazard_start_global": int(
                    idx[hazard_start_local]
                ),
                "event_step": int(
                    ep_steps[event_local]
                ),
                "baseline_raw_pitch": baseline,
                "event_raw_pitch": float(
                    ep_raw[event_local]
                ),
                "event_wrapped_abs_pitch": float(
                    wrapped_abs[event_global]
                ),
            }
        )

    return (
        events,
        normal_mask,
        non_inversion_episodes,
    )


def threshold_metrics(data, threshold):
    episodes = np.asarray(
        data["episode"],
        dtype=np.int64,
    )

    wrapped_abs = np.asarray(
        data["torso_pitch_abs"],
        dtype=np.float64,
    )

    steps = np.asarray(
        data["step"],
        dtype=np.int64,
    )

    (
        events,
        normal_mask,
        non_inversion_episodes,
    ) = inversion_events_and_normal_mask(data)

    detections = []
    lead_steps = []

    for event in events:
        ep = event["episode"]
        event_step = event["event_step"]

        start_step = max(
            0,
            event_step - PRE_EVENT_STEPS,
        )

        mask = (
            (episodes == ep)
            & (steps >= start_step)
            & (steps <= event_step)
        )

        crossing_steps = steps[mask][
            wrapped_abs[mask] > threshold
        ]

        detected = len(crossing_steps) > 0

        detections.append(detected)

        if detected:
            lead_steps.append(
                int(
                    event_step
                    - crossing_steps[0]
                )
            )

    if np.any(normal_mask):
        normal_step_false_positive_rate = float(
            np.mean(
                wrapped_abs[normal_mask]
                > threshold
            )
        )
    else:
        normal_step_false_positive_rate = np.nan

    nonflip_false_alarm = []

    for ep in non_inversion_episodes:
        mask = episodes == ep

        nonflip_false_alarm.append(
            bool(
                np.any(
                    wrapped_abs[mask]
                    > threshold
                )
            )
        )

    return {
        "inversion_episodes": len(events),
        "detection_rate": (
            float(np.mean(detections))
            if detections
            else np.nan
        ),
        "median_lead_steps": (
            float(np.median(lead_steps))
            if lead_steps
            else np.nan
        ),
        "mean_lead_steps": (
            float(np.mean(lead_steps))
            if lead_steps
            else np.nan
        ),
        "median_lead_seconds": (
            float(
                np.median(lead_steps)
                * CONTROL_DT
            )
            if lead_steps
            else np.nan
        ),
        "normal_step_false_positive_rate":
            normal_step_false_positive_rate,
        "nonflip_episode_false_alarm_rate": (
            float(
                np.mean(
                    nonflip_false_alarm
                )
            )
            if nonflip_false_alarm
            else np.nan
        ),
    }


(
    cal_events,
    cal_normal_mask,
    cal_nonflip,
) = inversion_events_and_normal_mask(cal)

(
    val_events,
    val_normal_mask,
    val_nonflip,
) = inversion_events_and_normal_mask(val)

print(
    "Calibration inversion episodes:",
    len(cal_events),
    "/",
    len(np.unique(cal["episode"])),
)

print(
    "Validation inversion episodes:",
    len(val_events),
    "/",
    len(np.unique(val["episode"])),
)

print(
    "Calibration non-inversion episodes:",
    len(cal_nonflip),
)

print(
    "Validation non-inversion episodes:",
    len(val_nonflip),
)

print()

rows = []

for threshold in CANDIDATE_THRESHOLDS:
    cal_metrics = threshold_metrics(
        cal,
        threshold,
    )

    val_metrics = threshold_metrics(
        val,
        threshold,
    )

    rows.append(
        {
            "threshold_rad":
                float(threshold),

            "threshold_deg":
                float(
                    np.degrees(threshold)
                ),

            "cal_detection_rate":
                cal_metrics[
                    "detection_rate"
                ],

            "val_detection_rate":
                val_metrics[
                    "detection_rate"
                ],

            "cal_median_lead_steps":
                cal_metrics[
                    "median_lead_steps"
                ],

            "val_median_lead_steps":
                val_metrics[
                    "median_lead_steps"
                ],

            "cal_median_lead_seconds":
                cal_metrics[
                    "median_lead_seconds"
                ],

            "val_median_lead_seconds":
                val_metrics[
                    "median_lead_seconds"
                ],

            "cal_normal_step_false_positive_rate":
                cal_metrics[
                    "normal_step_false_positive_rate"
                ],

            "val_normal_step_false_positive_rate":
                val_metrics[
                    "normal_step_false_positive_rate"
                ],

            "cal_nonflip_episode_false_alarm_rate":
                cal_metrics[
                    "nonflip_episode_false_alarm_rate"
                ],

            "val_nonflip_episode_false_alarm_rate":
                val_metrics[
                    "nonflip_episode_false_alarm_rate"
                ],
        }
    )


pitch_behavior_sweep = pd.DataFrame(
    rows
)

eligible = pitch_behavior_sweep[
    (
        pitch_behavior_sweep[
            "cal_detection_rate"
        ]
        >= MIN_DETECTION_RATE
    )
    & (
        pitch_behavior_sweep[
            "val_detection_rate"
        ]
        >= MIN_DETECTION_RATE
    )
    & (
        pitch_behavior_sweep[
            "cal_median_lead_steps"
        ]
        >= MIN_MEDIAN_LEAD_STEPS
    )
    & (
        pitch_behavior_sweep[
            "val_median_lead_steps"
        ]
        >= MIN_MEDIAN_LEAD_STEPS
    )
].copy()


if len(eligible) > 0:
    eligible[
        "worst_normal_step_false_positive_rate"
    ] = (
        eligible[
            [
                "cal_normal_step_false_positive_rate",
                "val_normal_step_false_positive_rate",
            ]
        ]
        .fillna(1.0)
        .max(axis=1)
    )

    best_false_positive = float(
        eligible[
            "worst_normal_step_false_positive_rate"
        ].min()
    )

    finalists = eligible[
        np.isclose(
            eligible[
                "worst_normal_step_false_positive_rate"
            ],
            best_false_positive,
        )
    ]

    recommended_row = (
        finalists
        .sort_values(
            "threshold_rad",
            ascending=False,
        )
        .iloc[0]
    )

    recommendation_method = (
        "Minimum worst-case normal-step false-positive rate "
        "among thresholds with at least 90% inversion detection "
        "and at least 5-step median warning lead in both "
        "calibration and validation"
    )

else:
    pitch_behavior_sweep[
        "minimum_detection_rate"
    ] = (
        pitch_behavior_sweep[
            [
                "cal_detection_rate",
                "val_detection_rate",
            ]
        ]
        .fillna(0.0)
        .min(axis=1)
    )

    pitch_behavior_sweep[
        "minimum_lead_steps"
    ] = (
        pitch_behavior_sweep[
            [
                "cal_median_lead_steps",
                "val_median_lead_steps",
            ]
        ]
        .fillna(0.0)
        .min(axis=1)
    )

    pitch_behavior_sweep[
        "worst_normal_step_false_positive_rate"
    ] = (
        pitch_behavior_sweep[
            [
                "cal_normal_step_false_positive_rate",
                "val_normal_step_false_positive_rate",
            ]
        ]
        .fillna(1.0)
        .max(axis=1)
    )

    pitch_behavior_sweep[
        "fallback_score"
    ] = (
        pitch_behavior_sweep[
            "minimum_detection_rate"
        ]
        - pitch_behavior_sweep[
            "worst_normal_step_false_positive_rate"
        ]
        + 0.01
        * np.minimum(
            pitch_behavior_sweep[
                "minimum_lead_steps"
            ],
            20,
        )
    )

    recommended_row = (
        pitch_behavior_sweep
        .sort_values(
            [
                "fallback_score",
                "threshold_rad",
            ],
            ascending=[
                False,
                False,
            ],
        )
        .iloc[0]
    )

    recommendation_method = (
        "Fallback balance because no threshold met "
        "the 90% detection and 5-step warning criteria "
        "in both datasets"
    )


recommended_pitch_threshold = float(
    recommended_row[
        "threshold_rad"
    ]
)

recommended_pitch_scale = (
    recommended_pitch_threshold
)

pitch_threshold = (
    recommended_pitch_threshold
)

pitch_scale = (
    recommended_pitch_scale
)


display(
    pitch_behavior_sweep
    .sort_values(
        [
            "val_normal_step_false_positive_rate",
            "cal_normal_step_false_positive_rate",
            "threshold_rad",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
    .head(20)
)


print()

print(
    "Recommendation method:"
)

print(
    recommendation_method
)

print()

print(
    "Recommended pitch threshold:",
    recommended_pitch_threshold,
    "rad",
)

print(
    "Recommended pitch threshold:",
    np.degrees(
        recommended_pitch_threshold
    ),
    "deg",
)

print(
    "Recommended pitch scale:",
    recommended_pitch_scale,
)

print()

print(
    "Calibration detection rate:",
    float(
        recommended_row[
            "cal_detection_rate"
        ]
    ),
)

print(
    "Validation detection rate:",
    float(
        recommended_row[
            "val_detection_rate"
        ]
    ),
)

print(
    "Calibration median warning lead:",
    float(
        recommended_row[
            "cal_median_lead_steps"
        ]
    ),
    "steps",
)

print(
    "Validation median warning lead:",
    float(
        recommended_row[
            "val_median_lead_steps"
        ]
    ),
    "steps",
)

print(
    "Calibration normal-step false-positive rate:",
    float(
        recommended_row[
            "cal_normal_step_false_positive_rate"
        ]
    ),
)

print(
    "Validation normal-step false-positive rate:",
    float(
        recommended_row[
            "val_normal_step_false_positive_rate"
        ]
    ),
)


pitch_behavior_sweep.to_csv(
    OUTPUT
    / "pitch_behavior_threshold_sweep.csv",
    index=False,
)


plt.figure(
    figsize=(10, 5)
)

plt.plot(
    pitch_behavior_sweep[
        "threshold_rad"
    ],
    pitch_behavior_sweep[
        "cal_detection_rate"
    ],
    label="Calibration inversion detection",
)

plt.plot(
    pitch_behavior_sweep[
        "threshold_rad"
    ],
    pitch_behavior_sweep[
        "val_detection_rate"
    ],
    label="Validation inversion detection",
)

plt.axvline(
    recommended_pitch_threshold,
    linestyle="--",
    label="Recommended threshold",
)

plt.xlabel(
    "Wrapped absolute pitch threshold (rad)"
)

plt.ylabel(
    "Inversion detection rate"
)

plt.ylim(
    -0.02,
    1.02,
)

plt.legend()

plt.tight_layout()

plt.show()


plt.figure(
    figsize=(10, 5)
)

plt.plot(
    pitch_behavior_sweep[
        "threshold_rad"
    ],
    pitch_behavior_sweep[
        "cal_normal_step_false_positive_rate"
    ],
    label="Calibration normal-step false positive",
)

plt.plot(
    pitch_behavior_sweep[
        "threshold_rad"
    ],
    pitch_behavior_sweep[
        "val_normal_step_false_positive_rate"
    ],
    label="Validation normal-step false positive",
)

plt.axvline(
    recommended_pitch_threshold,
    linestyle="--",
    label="Recommended threshold",
)

plt.xlabel(
    "Wrapped absolute pitch threshold (rad)"
)

plt.ylabel(
    "Normal-step false-positive rate"
)

plt.legend()

plt.tight_layout()

plt.show()


event_rows = []

for label, data, events in [
    (
        "calibration",
        cal,
        cal_events,
    ),
    (
        "validation",
        val,
        val_events,
    ),
]:
    episodes = np.asarray(
        data["episode"],
        dtype=np.int64,
    )

    steps = np.asarray(
        data["step"],
        dtype=np.int64,
    )

    wrapped_abs = np.asarray(
        data["torso_pitch_abs"],
        dtype=np.float64,
    )

    for event in events:
        ep = event["episode"]
        event_step = event["event_step"]

        start_step = max(
            0,
            event_step - PRE_EVENT_STEPS,
        )

        mask = (
            (episodes == ep)
            & (steps >= start_step)
            & (steps <= event_step)
        )

        crossing_steps = steps[mask][
            wrapped_abs[mask]
            > recommended_pitch_threshold
        ]

        first_crossing = (
            int(crossing_steps[0])
            if len(crossing_steps) > 0
            else np.nan
        )

        event_rows.append(
            {
                "dataset": label,
                "episode": ep,
                "event_step":
                    event_step,
                "first_threshold_crossing_step":
                    first_crossing,
                "warning_lead_steps": (
                    event_step
                    - first_crossing
                    if np.isfinite(
                        first_crossing
                    )
                    else np.nan
                ),
                "warning_lead_seconds": (
                    (
                        event_step
                        - first_crossing
                    )
                    * CONTROL_DT
                    if np.isfinite(
                        first_crossing
                    )
                    else np.nan
                ),
                "event_wrapped_abs_pitch":
                    event[
                        "event_wrapped_abs_pitch"
                    ],
            }
        )


pitch_flip_events = pd.DataFrame(
    event_rows
)

pitch_flip_events.to_csv(
    OUTPUT
    / "pitch_flip_event_detection.csv",
    index=False,
)

display(
    pitch_flip_events.head(20)
)

print()

print(
    "Saved:",
    OUTPUT
    / "pitch_behavior_threshold_sweep.csv",
)

print(
    "Saved:",
    OUTPUT
    / "pitch_flip_event_detection.csv",
)